In [0]:
%python
%pip install openpyxl

Looking in indexes: [REDACTED]
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
%python
%restart_python

In [0]:
%python
import pandas as pd

source_file = "/Volumes/retail_lakehouse/retail/retail_files/raw/online_retail_II.xlsx"

xls = pd.ExcelFile(source_file)

print(xls.sheet_names)

['Year 2009-2010', 'Year 2010-2011']


In [0]:
%python
df_1 = pd.read_excel(
    source_file,
    sheet_name = 'Year 2009-2010'
)

df_2 = pd.read_excel(
    source_file,
    sheet_name = 'Year 2010-2011'
)

df = pd.concat([df_1, df_2],ignore_index= True)

df["StockCode"] = df["StockCode"].astype("string")

display(df.head())

Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01T07:45:00.000Z,6.95,13085.0,United Kingdom
489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01T07:45:00.000Z,6.75,13085.0,United Kingdom
489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01T07:45:00.000Z,6.75,13085.0,United Kingdom
489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01T07:45:00.000Z,2.1,13085.0,United Kingdom
489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01T07:45:00.000Z,1.25,13085.0,United Kingdom


In [0]:
%python
df.shape

(1067371, 8)

In [0]:
%python
df.columns

Index(['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate',
       'Price', 'Customer ID', 'Country'],
      dtype='object')

In [0]:
%python
df.isnull().sum()

Invoice             0
StockCode           0
Description      4382
Quantity            0
InvoiceDate         0
Price               0
Customer ID    243007
Country             0
dtype: int64

In [0]:
%python
df.dtypes

Invoice                object
StockCode      string[python]
Description            object
Quantity                int64
InvoiceDate    datetime64[ns]
Price                 float64
Customer ID           float64
Country                object
dtype: object

In [0]:
%python
df['Invoice'] = df['Invoice'].astype("string")
df["Customer ID"] = df["Customer ID"].astype("string")

df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"],errors="coerce")


In [0]:
%python
df['InvoiceMonth'] = df['InvoiceDate'].dt.strftime('%Y-%m')

In [0]:
%python
df['Customer ID'].rename('CustomerID', inplace=True)

0          13085.0
1          13085.0
2          13085.0
3          13085.0
4          13085.0
            ...   
1067366    12680.0
1067367    12680.0
1067368    12680.0
1067369    12680.0
1067370    12680.0
Name: CustomerID, Length: 1067371, dtype: string

In [0]:
%python
df.dtypes

Invoice         string[python]
StockCode       string[python]
Description             object
Quantity                 int64
InvoiceDate     datetime64[ns]
Price                  float64
Customer ID     string[python]
Country                 object
InvoiceMonth            object
dtype: object

In [0]:
%python
df['Description'] = df['Description'].astype("string")
df['Country'] = df['Country'].astype("string")

In [0]:
%python
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,InvoiceMonth
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,2009-12
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,2009-12
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,2009-12
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,2009-12
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,2009-12


In [0]:
import os
output_dir = "/Volumes/retail_lakehouse/retail/retail_files/incoming"

os.makedirs(output_dir, exist_ok = True)

for month, monthly_df in df.groupby('InvoiceMonth'):
    if pd.isna(month):
        continue
    output_file = f"{output_dir}/retail_{month}.csv"

    monthly_df.drop(columns = ["InvoiceMonth"]).to_csv(output_file, index=False)
    
    print(f"{month}: {len(monthly_df):,} rows")

2009-12: 45,228 rows
2010-01: 31,555 rows
2010-02: 29,388 rows
2010-03: 41,511 rows
2010-04: 34,057 rows
2010-05: 35,323 rows
2010-06: 39,983 rows
2010-07: 33,383 rows
2010-08: 33,306 rows
2010-09: 42,091 rows
2010-10: 59,098 rows
2010-11: 78,015 rows
2010-12: 65,004 rows
2011-01: 35,147 rows
2011-02: 27,707 rows
2011-03: 36,748 rows
2011-04: 29,916 rows
2011-05: 37,030 rows
2011-06: 36,874 rows
2011-07: 39,518 rows
2011-08: 35,284 rows
2011-09: 50,226 rows
2011-10: 60,742 rows
2011-11: 84,711 rows
2011-12: 25,526 rows


In [0]:
display(
    dbutils.fs.ls(output_dir)
)

path,name,size,modificationTime
dbfs:/Volumes/retail_lakehouse/retail/retail_files/incoming/retail_2009-12.csv,retail_2009-12.csv,4023069,1791457708000
dbfs:/Volumes/retail_lakehouse/retail/retail_files/incoming/retail_2010-01.csv,retail_2010-01.csv,2798771,1791457708000
dbfs:/Volumes/retail_lakehouse/retail/retail_files/incoming/retail_2010-02.csv,retail_2010-02.csv,2612287,1791457708000
dbfs:/Volumes/retail_lakehouse/retail/retail_files/incoming/retail_2010-03.csv,retail_2010-03.csv,3697902,1791457709000
dbfs:/Volumes/retail_lakehouse/retail/retail_files/incoming/retail_2010-04.csv,retail_2010-04.csv,3033096,1791457709000
dbfs:/Volumes/retail_lakehouse/retail/retail_files/incoming/retail_2010-05.csv,retail_2010-05.csv,3136698,1791457710000
dbfs:/Volumes/retail_lakehouse/retail/retail_files/incoming/retail_2010-06.csv,retail_2010-06.csv,3552525,1791457710000
dbfs:/Volumes/retail_lakehouse/retail/retail_files/incoming/retail_2010-07.csv,retail_2010-07.csv,2973236,1791457710000
dbfs:/Volumes/retail_lakehouse/retail/retail_files/incoming/retail_2010-08.csv,retail_2010-08.csv,2968974,1791457711000
dbfs:/Volumes/retail_lakehouse/retail/retail_files/incoming/retail_2010-09.csv,retail_2010-09.csv,3766130,1791457711000


---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-5909639879582826>, line 1
----> 1 df.head()

NameError: name 'df' is not defined